In [1]:
%run training_setup.ipynb

c:\Users\Nourhan Yehia\Desktop\Jupyter\xray_pneumonia_classifier
4172 1044 624
torch.Size([32, 1, 224, 224])
torch.Size([32])
tensor([1, 1, 1, 1, 1, 1, 1, 0, 1, 1])


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)

model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

c:\Users\Nourhan Yehia\.conda\envs\cnn_xray\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Nourhan Yehia\.conda\envs\cnn_xray\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

In [5]:
class_counts = train_df["class"].value_counts()

n_normal = class_counts["NORMAL"]
n_pneu   = class_counts["PNEUMONIA"]

total = n_normal + n_pneu

w_normal = total / (2 * n_normal)
w_pneu   = total / (2 * n_pneu)

class_weights = torch.tensor([w_normal, w_pneu], dtype=torch.float32).to(device)

class_weights

tensor([1.9441, 0.6731], device='cuda:0')

In [6]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam([
    {"params": model.fc.parameters(), "lr": 1e-3},
    {"params": model.layer4.parameters(), "lr": 1e-4}
])

In [7]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total

In [8]:
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [9]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 40)

Epoch [1/5]
Train Loss: 0.1384 | Train Acc: 0.9420
Val   Loss: 0.0640 | Val   Acc: 0.9789
----------------------------------------
Epoch [2/5]
Train Loss: 0.0328 | Train Acc: 0.9866
Val   Loss: 0.0804 | Val   Acc: 0.9818
----------------------------------------
Epoch [3/5]
Train Loss: 0.0116 | Train Acc: 0.9964
Val   Loss: 0.0542 | Val   Acc: 0.9895
----------------------------------------
Epoch [4/5]
Train Loss: 0.0059 | Train Acc: 0.9986
Val   Loss: 0.0510 | Val   Acc: 0.9837
----------------------------------------
Epoch [5/5]
Train Loss: 0.0041 | Train Acc: 0.9981
Val   Loss: 0.0890 | Val   Acc: 0.9799
----------------------------------------


In [10]:
import torch.nn.functional as F

model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

thr = 0.6  

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        probs = F.softmax(outputs, dim=1)
        p_pneu = probs[:, 1]

        preds = (p_pneu >= thr).long()

        total += labels.size(0)
        correct += (preds == labels).sum().item()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.7965


In [11]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds, target_names=["NORMAL", "PNEUMONIA"]))

              precision    recall  f1-score   support

      NORMAL       0.99      0.46      0.63       234
   PNEUMONIA       0.76      1.00      0.86       390

    accuracy                           0.80       624
   macro avg       0.87      0.73      0.74       624
weighted avg       0.84      0.80      0.77       624



In [12]:
torch.save(model.state_dict(), "resnet18_final.pth")